In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["# SignVoice — Data Exploration\nVisualize keypoints, mel spectrograms, and dataset statistics."]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\n",
    "sys.path.insert(0, '..')\n",
    "\n",
    "import json\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import librosa\n",
    "import librosa.display\n",
    "from pathlib import Path\n",
    "from collections import Counter\n",
    "\n",
    "print('Imports OK')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 1. Dataset Statistics"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "kp_dir  = Path('../data/processed/keypoints')\n",
    "mel_dir = Path('../data/processed/mels')\n",
    "\n",
    "print(f'Keypoint files : {len(list(kp_dir.glob(\"*.npy\")))}')\n",
    "print(f'Mel files      : {len(list(mel_dir.glob(\"*.npy\")))}')\n",
    "print()\n",
    "\n",
    "for split in ['train', 'val', 'test']:\n",
    "    path = Path(f'../data/processed/{split}_manifest.json')\n",
    "    if path.exists():\n",
    "        with open(path) as f:\n",
    "            data = json.load(f)\n",
    "        glosses = set(s['gloss'] for s in data)\n",
    "        sources = Counter(s.get('source', 'unknown') for s in data)\n",
    "        print(f'{split:>5} : {len(data):>4} samples | '\n",
    "              f'{len(glosses)} glosses | sources={dict(sources)}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 2. Gloss Distribution"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "with open('../data/processed/train_manifest.json') as f:\n",
    "    train_data = json.load(f)\n",
    "\n",
    "gloss_counts  = Counter(s['gloss'] for s in train_data)\n",
    "source_counts = Counter(s.get('source','unknown') for s in train_data)\n",
    "\n",
    "fig, axes = plt.subplots(1, 2, figsize=(16, 4))\n",
    "\n",
    "glosses = list(gloss_counts.keys())\n",
    "counts  = list(gloss_counts.values())\n",
    "axes[0].bar(glosses, counts, color='steelblue', edgecolor='white')\n",
    "axes[0].set_xlabel('Gloss')\n",
    "axes[0].set_ylabel('Samples')\n",
    "axes[0].set_title('Training samples per gloss')\n",
    "plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=45, ha='right')\n",
    "\n",
    "axes[1].pie(source_counts.values(), labels=source_counts.keys(),\n",
    "            autopct='%1.1f%%', colors=['steelblue','coral'])\n",
    "axes[1].set_title('Data source distribution')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()\n",
    "print(f'Gloss counts : {dict(gloss_counts)}')\n",
    "print(f'Sources      : {dict(source_counts)}')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 3. Visualize Keypoints"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "kp_files = sorted(Path('../data/processed/keypoints').glob('*.npy'))\n",
    "kp       = np.load(kp_files[0])\n",
    "print(f'Keypoint shape : {kp.shape}  (frames x features)')\n",
    "\n",
    "fig, axes = plt.subplots(1, 2, figsize=(14, 4))\n",
    "\n",
    "axes[0].imshow(kp.T, aspect='auto', origin='lower', cmap='viridis')\n",
    "axes[0].set_xlabel('Frame')\n",
    "axes[0].set_ylabel('Feature index')\n",
    "axes[0].set_title(f'Keypoint heatmap {kp.shape}')\n",
    "axes[0].axhline(63,  color='red',    linestyle='--', linewidth=1, label='left/right hand')\n",
    "axes[0].axhline(126, color='yellow', linestyle='--', linewidth=1, label='hand/pose')\n",
    "axes[0].axhline(171, color='white',  linestyle='--', linewidth=1, label='pose/face')\n",
    "axes[0].legend(fontsize=8)\n",
    "plt.colorbar(axes[0].images[0], ax=axes[0])\n",
    "\n",
    "variance = np.var(kp, axis=0)\n",
    "axes[1].plot(variance, color='coral', linewidth=0.8)\n",
    "axes[1].set_xlabel('Feature index')\n",
    "axes[1].set_ylabel('Variance')\n",
    "axes[1].set_title('Feature variance — which features move most')\n",
    "axes[1].axvline(63,  color='red',    linestyle='--', linewidth=0.8)\n",
    "axes[1].axvline(126, color='yellow', linestyle='--', linewidth=0.8)\n",
    "axes[1].axvline(171, color='orange', linestyle='--', linewidth=0.8)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 4. Compare Keypoints Across Signs"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "with open('../data/processed/train_manifest.json') as f:\n",
    "    train_data = json.load(f)\n",
    "\n",
    "glosses = sorted(set(s['gloss'] for s in train_data))\n",
    "print(f'Signs in dataset: {glosses}')\n",
    "\n",
    "fig, axes = plt.subplots(2, 5, figsize=(20, 6))\n",
    "axes = axes.flatten()\n",
    "\n",
    "for i, gloss in enumerate(glosses[:10]):\n",
    "    samples = [s for s in train_data if s['gloss'] == gloss]\n",
    "    if not samples:\n",
    "        continue\n",
    "    kp = np.load(samples[0]['keypoint_file'])\n",
    "    axes[i].imshow(kp.T, aspect='auto', origin='lower',\n",
    "                   cmap='viridis')\n",
    "    axes[i].set_title(f'{gloss} ({kp.shape[0]} frames)')\n",
    "    axes[i].set_xlabel('Frame')\n",
    "    axes[i].set_ylabel('Feature')\n",
    "\n",
    "plt.suptitle('Keypoint patterns per sign', fontsize=12)\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 5. Visualize Mel Spectrogram"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "with open('../data/processed/train_manifest.json') as f:\n",
    "    train_data = json.load(f)\n",
    "\n",
    "glosses = sorted(set(s['gloss'] for s in train_data))\n",
    "fig, axes = plt.subplots(2, 5, figsize=(20, 6))\n",
    "axes = axes.flatten()\n",
    "\n",
    "for i, gloss in enumerate(glosses[:10]):\n",
    "    samples = [s for s in train_data\n",
    "               if s['gloss'] == gloss and 'mel_file' in s]\n",
    "    if not samples:\n",
    "        continue\n",
    "    mel = np.load(samples[0]['mel_file'])\n",
    "    librosa.display.specshow(\n",
    "        mel, sr=22050, hop_length=256,\n",
    "        x_axis='time', y_axis='mel',\n",
    "        ax=axes[i], cmap='magma'\n",
    "    )\n",
    "    axes[i].set_title(f\"Mel: '{gloss}'\")\n",
    "\n",
    "plt.suptitle('Target mel spectrograms per sign', fontsize=12)\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 6. Sequence Length Distribution"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "with open('../data/processed/train_manifest.json') as f:\n",
    "    train_data = json.load(f)\n",
    "\n",
    "kp_lengths  = [s['kp_frames'] for s in train_data]\n",
    "mel_lengths = [s['mel_frames'] for s in train_data if 'mel_frames' in s]\n",
    "\n",
    "fig, axes = plt.subplots(1, 2, figsize=(12, 4))\n",
    "\n",
    "axes[0].hist(kp_lengths, bins=30, color='steelblue', edgecolor='white')\n",
    "axes[0].set_xlabel('Frames')\n",
    "axes[0].set_ylabel('Count')\n",
    "axes[0].set_title(\n",
    "    f'Keypoint lengths\\n'\n",
    "    f'mean={np.mean(kp_lengths):.1f} '\n",
    "    f'min={min(kp_lengths)} '\n",
    "    f'max={max(kp_lengths)}'\n",
    ")\n",
    "\n",
    "axes[1].hist(mel_lengths, bins=20, color='coral', edgecolor='white')\n",
    "axes[1].set_xlabel('Frames')\n",
    "axes[1].set_ylabel('Count')\n",
    "axes[1].set_title(\n",
    "    f'Mel lengths\\n'\n",
    "    f'mean={np.mean(mel_lengths):.1f}'\n",
    ")\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": ["## 7. Google ASL vs WLASL Comparison"]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "with open('../data/processed/train_manifest.json') as f:\n",
    "    train_data = json.load(f)\n",
    "\n",
    "glosses = sorted(set(s['gloss'] for s in train_data))\n",
    "\n",
    "print('Dataset composition per sign:')\n",
    "print(f'{\"Sign\":15s} | Google ASL | WLASL | Total')\n",
    "print('-' * 45)\n",
    "\n",
    "for g in glosses:\n",
    "    asl   = sum(1 for s in train_data\n",
    "                if s['gloss'] == g and s.get('source') == 'google_asl')\n",
    "    wlasl = sum(1 for s in train_data\n",
    "                if s['gloss'] == g and s.get('source') == 'wlasl')\n",
    "    total = asl + wlasl\n",
    "    bar   = '█' * (asl // 5) + '░' * (wlasl // 5)\n",
    "    print(f'{g:15s} | {asl:>10} | {wlasl:>5} | {total:>5}  {bar}')"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "SignVoice (venv311)",
   "language": "python",
   "name": "signvoice"
  },
  "language_info": {
   "name": "python",
   "version": "3.11.9"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}